# Human and LLM Judges: Four Metrics

::::::::{div}
:class: progress-analysis-app

:::::::{div}
:class: progress2-section progress2-input

## Input

Each question now has a **separate fitted model** for each of the four metrics. Two score mappings give **40 fits** in total. The automatic judge is Llama 3.1 8B; candidate answers come from GPT 4o and Llama 3.1 405B.

| Human dimension | Automatic metric | Meaning | Proxy or mismatch |
| --- | --- | --- | --- |
| Faithfulness | DeepEval Faithfulness | Claims supported by retrieved context | Same target; automatic claim checking differs from human grading |
| Appropriateness | DeepEval AnswerRelevancy | Relevance to the question | Proxy: relevance is narrower than overall appropriateness |
| Naturalness | Custom G Eval | Fluent and natural wording | Rubric approximation without log probability weighting |
| Completeness | Custom G Eval | Coverage of what the answer needs | Rubric approximation; no reference answer supplied |

| Question | Dedicated model |
| --- | --- |
| Q1 | Grader type + generator; no interaction or length term |
| Q2 | Grader type + generator + their interaction |
| Q3 | Grader type + generator + estimated human effect scale |
| Q4 | Grader type + generator + question + automatic judge × question |
| Q5 | Grader type + generator + length + automatic judge × length |

| Experiment | Setup |
| --- | --- |
| Planned sample | 50 questions × 2 answers × 4 metrics = 400 automatic scores |
| Human ratings | Released MTRAG annotations; categories 1 to 4; seven annotators |
| Automatic ratings | DeepEval 4.3.1; scores 0 to 1; temperature 0 |
| Judge input | Question, answer and retrieved context; generator identity and human ratings hidden |
| Available domains | ClapNQ, FiQA and Govt; the fourth collection has no completed scores |

| Metric | Valid answers / 100 | Questions | Failed or timed out | Not run |
| --- | --- | --- | --- | --- |
| Faithfulness | 44 | 29 | 32 | 24 |
| Appropriateness proxy | 63 | 36 | 12 | 25 |
| Naturalness | 75 | 38 | 0 | 25 |
| Completeness | 75 | 38 | 0 | 25 |

### Scope

Failed and missing scores are omitted, not coded as zero. The metric subsets differ. Intervals are pointwise 95% intervals, not simultaneous guarantees across all comparisons. The regression models include domain, conversation and answer adjustments; these do not replace missing repeated generations or additional grader identities. Q3 estimates the human effect scale; the other models use a fixed prior scale for human effects.

::::::{dropdown} Score mapping and priors


| Choice | Specification |
| --- | --- |
| Primary mapping | Automatic categories 1 to 4, boundaries 1/6, 1/2, 5/6 |
| Alternative mapping | Boundaries 1/4, 1/2, 3/4; boundary values enter the higher category |
| Descriptive display | Human mean transformed as (mean − 1) / 3 |
| Fixed effects | Normal(0,1); centered domain and human effects |
| Group effects | Centered standard normal effects multiplied by HalfNormal(1) scales |
| Cutpoints | First: Normal(−1,0.2); positive gaps: LogNormal(−0.5,0.3) + 0.3 |
| Sampling | Four chains per fit, at least 1,000 warmup and 1,000 retained draws per chain; seed 20261102 |

The category mappings are assumptions, not a validated shared rubric. The first cutpoint is recentered for four categories. The paper uses different data and coding; the adaptation is shown separately in every question.

The former joint fits remain on disk for provenance. The figures and conclusions below use the dedicated question fits. No new LLM evaluations were run.

::::::

::::::{dropdown} Coverage figure


:::::{div}
:class: research-plots

```{figure} figures/progress2/coverage-faithfulness.svg
:alt: Faithfulness. Coverage out of 100 planned answers per metric.
```
```{figure} figures/progress2/coverage-appropriateness.svg
:alt: Appropriateness proxy. Coverage out of 100 planned answers per metric.
```
```{figure} figures/progress2/coverage-naturalness.svg
:alt: Naturalness. Coverage out of 100 planned answers per metric.
```
```{figure} figures/progress2/coverage-completeness.svg
:alt: Completeness. Coverage out of 100 planned answers per metric.
```
:::::

Coverage out of 100 planned answers per metric.

::::::

:::::::

:::::::{div}
:class: progress2-section progress2-q1

## Q1. How does the autograder compare with humans?

### Model

The paper compares grader and generator main effects. This fit contains no generator interaction, question deviation or length term.


:::::{div}
:class: equation-sequence

**Paper specification** (notation simplified):

$$
\begin{aligned}
\phi_i&=\beta_0+\beta_1 x_i^{\mathrm{grader}}+\beta_2 x_i^{\mathrm{LLM}},\\
S_i&\sim\operatorname{OrderedLogistic}(\phi_i,\mathbf c).
\end{aligned}
$$
:::::

:::::{div}
:class: equation-sequence

**Separately fitted model for this question**, for each metric $m$:

$$
\eta_{im}=B_{im}+\beta_{Am}A_i+\beta_{Lm}L_i+(1-A_i)h_{j(i)m}.
$$

The rating likelihood is:

$$
Y_{im}\sim\operatorname{OrderedLogistic}(\eta_{im},\mathbf c_m).
$$
:::::

**Quantity answering this question:**

$$
\Delta_{AH,m}=\beta_{Am},\qquad\Delta_{L-G,m}=\beta_{Lm}.
$$

| Symbol | Meaning |
| --- | --- |
| $\beta_{Am}$ | Automatic minus average human contrast |
| $\beta_{Lm}$ | Llama minus GPT contrast, common to both grader types | 
| $i,m$ | Annotation and metric; each metric has its own coefficients and cutpoints |
| $Y_{im},\mathbf c_m$ | Category 1 to 4 and three ordered cutpoints |
| $A_i,L_i$ | Automatic = 1, human = 0; Llama = +0.5, GPT = −0.5 |
| $B_{im}$ | $\beta_{0m}+d_{\mathrm{domain}(i),m}+v_{\mathrm{conversation}(i),m}+r_{\mathrm{answer}(i),m}$ |

$B_{im}$ adjusts for domain and repeated annotations of answers within conversations. Human effects use $h_{jm}=\xi_{jm}-\bar\xi_m$, $\xi_{jm}\sim\mathcal N(0,1)$ with a fixed prior scale; Q3 instead estimates that scale.

### Results


:::::{div}
:class: research-plots

```{figure} figures/progress2/q1-effects-faithfulness.svg
:alt: Faithfulness. Ochre squares: automatic minus human. Blue diamonds: generator contrast. Bars: 95% credible intervals.
```
```{figure} figures/progress2/q1-effects-appropriateness.svg
:alt: Appropriateness proxy. Ochre squares: automatic minus human. Blue diamonds: generator contrast. Bars: 95% credible intervals.
```
```{figure} figures/progress2/q1-effects-naturalness.svg
:alt: Naturalness. Ochre squares: automatic minus human. Blue diamonds: generator contrast. Bars: 95% credible intervals.
```
```{figure} figures/progress2/q1-effects-completeness.svg
:alt: Completeness. Ochre squares: automatic minus human. Blue diamonds: generator contrast. Bars: 95% credible intervals.
```
:::::

Ochre squares: automatic minus human. Blue diamonds: generator contrast. Bars: 95% credible intervals.

::::::{dropdown} Numerical estimates and sensitivity


| Metric | Primary median [95% CrI] | Alternative median [95% CrI] |
| --- | --- | --- |
| Faithfulness | -0.64 [-1.27, -0.01] | -0.35 [-0.98, +0.32] |
| Appropriateness proxy | -1.37 [-1.99, -0.76] | -1.00 [-1.62, -0.36] |
| Naturalness | -3.00 [-3.64, -2.39] | -0.47 [-1.18, +0.21] |
| Completeness | -1.78 [-2.32, -1.27] | -1.99 [-2.54, -1.47] |

:::::{div}
:class: research-plots

```{figure} figures/progress2/sensitivity-faithfulness.svg
:alt: Faithfulness. Severity uses the Q1 fits; interaction uses the Q2 fits. Green circles: primary mapping. Ochre squares: alternative mapping.
```
```{figure} figures/progress2/sensitivity-appropriateness.svg
:alt: Appropriateness proxy. Severity uses the Q1 fits; interaction uses the Q2 fits. Green circles: primary mapping. Ochre squares: alternative mapping.
```
```{figure} figures/progress2/sensitivity-naturalness.svg
:alt: Naturalness. Severity uses the Q1 fits; interaction uses the Q2 fits. Green circles: primary mapping. Ochre squares: alternative mapping.
```
```{figure} figures/progress2/sensitivity-completeness.svg
:alt: Completeness. Severity uses the Q1 fits; interaction uses the Q2 fits. Green circles: primary mapping. Ochre squares: alternative mapping.
```
:::::

Severity uses the Q1 fits; interaction uses the Q2 fits. Green circles: primary mapping. Ochre squares: alternative mapping.

::::::

::::::{dropdown} Raw scores


:::::{div}
:class: research-plots

```{figure} figures/progress2/q1-observed-faithfulness.svg
:alt: Faithfulness. Green points: human means. Ochre points: automatic scores. Each point is one answer.
```
```{figure} figures/progress2/q1-observed-appropriateness.svg
:alt: Appropriateness proxy. Green points: human means. Ochre points: automatic scores. Each point is one answer.
```
```{figure} figures/progress2/q1-observed-naturalness.svg
:alt: Naturalness. Green points: human means. Ochre points: automatic scores. Each point is one answer.
```
```{figure} figures/progress2/q1-observed-completeness.svg
:alt: Completeness. Green points: human means. Ochre points: automatic scores. Each point is one answer.
```
:::::

Green points: human means. Ochre points: automatic scores. Each point is one answer.

::::::

### Interpretation

The automatic contrast combines judge, rubric and mapping differences. Humans are a comparison group, not established ground truth.

### Conclusion

- **Faithfulness:** Conclusion depends on mapping.
- **Appropriateness proxy:** Lower automatic categories under both mappings.
- **Naturalness:** Conclusion depends on mapping.
- **Completeness:** Lower automatic categories under both mappings.

:::::::

:::::::{div}
:class: progress2-section progress2-q2

## Q2. Does the judge favor Llama family answers?

### Model

The paper adds grader × generator interactions. This fit estimates the automatic versus human difference in the Llama versus GPT contrast.


:::::{div}
:class: equation-sequence

**Paper specification** (notation simplified):

$$
\begin{aligned}
\phi_i&=\beta_0+\beta_{1,g_i}+\beta_2x_i^{\mathrm{LLM}}+\beta_{3,g_i,\ell_i},\\
S_i&\sim\operatorname{OrderedLogistic}(\phi_i,\mathbf c).
\end{aligned}
$$
:::::

:::::{div}
:class: equation-sequence

**Separately fitted model for this question**, for each metric $m$:

$$
\begin{aligned}
\eta_{im}={}&B_{im}+\beta_{Am}A_i+\beta_{Lm}L_i\\
&+\beta_{ALm}A_iL_i+(1-A_i)h_{j(i)m}.
\end{aligned}
$$

The rating likelihood is:

$$
Y_{im}\sim\operatorname{OrderedLogistic}(\eta_{im},\mathbf c_m).
$$
:::::

:::::{div}
:class: equation-sequence

**Quantity answering this question:**

$$
\begin{aligned}
\Delta_{L-G,m}^{H}&=\beta_{Lm},\\
\Delta_{L-G,m}^{A}&=\beta_{Lm}+\beta_{ALm},\\
\Delta_{L-G,m}^{A}-\Delta_{L-G,m}^{H}&=\beta_{ALm}.
\end{aligned}
$$
:::::

| Symbol | Meaning |
| --- | --- |
| $\beta_{Lm}$ | Generator contrast among humans |
| $\beta_{ALm}$ | Additional generator contrast for the automatic judge |
| $g_i,\ell_i$ in the paper | Grader and generator identities | 
| $i,m$ | Annotation and metric; each metric has its own coefficients and cutpoints |
| $Y_{im},\mathbf c_m$ | Category 1 to 4 and three ordered cutpoints |
| $A_i,L_i$ | Automatic = 1, human = 0; Llama = +0.5, GPT = −0.5 |
| $B_{im}$ | $\beta_{0m}+d_{\mathrm{domain}(i),m}+v_{\mathrm{conversation}(i),m}+r_{\mathrm{answer}(i),m}$ |

$B_{im}$ adjusts for domain and repeated annotations of answers within conversations. Human effects use $h_{jm}=\xi_{jm}-\bar\xi_m$, $\xi_{jm}\sim\mathcal N(0,1)$ with a fixed prior scale; Q3 instead estimates that scale.

### Results


:::::{div}
:class: research-plots

```{figure} figures/progress2/q2-interaction-faithfulness.svg
:alt: Faithfulness. Green circles: human generator contrast. Ochre squares: automatic contrast. Blue diamonds: their difference.
```
```{figure} figures/progress2/q2-interaction-appropriateness.svg
:alt: Appropriateness proxy. Green circles: human generator contrast. Ochre squares: automatic contrast. Blue diamonds: their difference.
```
```{figure} figures/progress2/q2-interaction-naturalness.svg
:alt: Naturalness. Green circles: human generator contrast. Ochre squares: automatic contrast. Blue diamonds: their difference.
```
```{figure} figures/progress2/q2-interaction-completeness.svg
:alt: Completeness. Green circles: human generator contrast. Ochre squares: automatic contrast. Blue diamonds: their difference.
```
:::::

Green circles: human generator contrast. Ochre squares: automatic contrast. Blue diamonds: their difference.

| Metric | Primary median [95% CrI] | Alternative median [95% CrI] |
| --- | --- | --- |
| Faithfulness | -0.29 [-1.38, +0.78] | -0.13 [-1.24, +0.90] |
| Appropriateness proxy | -0.28 [-1.27, +0.69] | -0.46 [-1.48, +0.55] |
| Naturalness | +0.47 [-0.47, +1.36] | +0.44 [-0.76, +1.58] |
| Completeness | +0.08 [-0.80, +0.96] | +0.13 [-0.77, +1.04] |

### Interpretation

Only one judge and two generators are observed. This interaction does not establish a general self bias effect. An interval containing zero does not prove equality.

### Conclusion

- No clear Llama family preference is resolved: all four interaction intervals include zero under both mappings.

:::::::

:::::::{div}
:class: progress2-section progress2-q3

## Q3. How much do human graders differ?


:::::{admonition} Design limitation: this is not the paper's full Q3 comparison
:class: warning

**Paper:** Section 3.3 compares several human and automatic graders using separate group means and scales.

**Our data:** seven human annotators and one automatic judge, Llama 3.1 8B.

**Limit:** we estimate variation among these humans and this judge's contrast against their average, not variation across automatic judges or a general human versus LLM difference.
:::::

### Model

The paper estimates variation within human and automatic grader populations. We estimate the human spread across seven annotators; the single automatic judge receives a fixed contrast, not a population variance.


:::::{div}
:class: equation-sequence

**Paper specification** (notation simplified):

$$
\begin{aligned}
\phi_i&=\beta_0+b_{g_i}+\beta_2x_i^{\mathrm{LLM}},\\
S_i&\sim\operatorname{OrderedLogistic}(\phi_i,\mathbf c),\\
b_g\mid t(g)&\sim\mathcal N(\mu_{t(g)},\sigma_{t(g)}^2).
\end{aligned}
$$
:::::

:::::{div}
:class: equation-sequence

**Separately fitted model for this question**, for each metric $m$:

$$
\begin{aligned}
\eta_{im}&=B_{im}+\beta_{Am}A_i+\beta_{Lm}L_i+(1-A_i)h_{j(i)m},\\
h_{jm}&=\sigma_{Hm}(\xi_{jm}-\bar\xi_m),\\
\xi_{jm}&\overset{\mathrm{iid}}{\sim}\mathcal N(0,1),\qquad
\sigma_{Hm}\sim\operatorname{HalfNormal}(1).
\end{aligned}
$$

The rating likelihood is:

$$
Y_{im}\sim\operatorname{OrderedLogistic}(\eta_{im},\mathbf c_m).
$$
:::::

**Quantity answering this question:**

$$
h_{jm},\qquad\sigma_{Hm},\qquad\Delta_{AH,m}=\beta_{Am}.
$$

| Symbol | Meaning |
| --- | --- |
| $h_{jm}$ | Human deviation from the average human; $\sum_jh_{jm}=0$ |
| $\sigma_{Hm}$ | Estimated spread of human grading effects |
| $\bar\xi_m$ | Mean of the seven standardized human effects |
| $t(g)$ in the paper | Human or automatic grader population | 
| $i,m$ | Annotation and metric; each metric has its own coefficients and cutpoints |
| $Y_{im},\mathbf c_m$ | Category 1 to 4 and three ordered cutpoints |
| $A_i,L_i$ | Automatic = 1, human = 0; Llama = +0.5, GPT = −0.5 |
| $B_{im}$ | $\beta_{0m}+d_{\mathrm{domain}(i),m}+v_{\mathrm{conversation}(i),m}+r_{\mathrm{answer}(i),m}$ |

$B_{im}$ adjusts for domain and repeated annotations of answers within conversations. Q3 estimates human variation through partial pooling. The scale prior is HalfNormal(1), an explicit adaptation of the paper.

### Results


| Metric | Above average in both fits | Below average in both fits |
| --- | --- | --- |
| Faithfulness | H02 | H07 |
| Appropriateness proxy | None resolved | None resolved |
| Naturalness | None resolved | None resolved |
| Completeness | H02, H06 | H07 |

:::::{div}
:class: research-plots

```{figure} figures/progress2/q3-graders-faithfulness.svg
:alt: Faithfulness. Green circles: humans. Ochre square: the automatic judge. Zero: average human; bars: 95% credible intervals.
```
```{figure} figures/progress2/q3-graders-appropriateness.svg
:alt: Appropriateness proxy. Green circles: humans. Ochre square: the automatic judge. Zero: average human; bars: 95% credible intervals.
```
```{figure} figures/progress2/q3-graders-naturalness.svg
:alt: Naturalness. Green circles: humans. Ochre square: the automatic judge. Zero: average human; bars: 95% credible intervals.
```
```{figure} figures/progress2/q3-graders-completeness.svg
:alt: Completeness. Green circles: humans. Ochre square: the automatic judge. Zero: average human; bars: 95% credible intervals.
```
:::::

Green circles: humans. Ochre square: the automatic judge. Zero: average human; bars: 95% credible intervals.

::::::{dropdown} Exact human effects


| Metric / grader | Primary median [95% CrI] | Alternative median [95% CrI] |
| --- | --- | --- |
| Faithfulness / H01 | +0.37 [-0.43, +1.34] | +0.33 [-0.43, +1.30] |
| Faithfulness / H02 | +0.72 [+0.09, +1.43] | +0.66 [+0.04, +1.42] |
| Faithfulness / H03 | +0.19 [-0.60, +1.12] | +0.16 [-0.60, +1.06] |
| Faithfulness / H04 | +0.01 [-1.14, +1.24] | +0.04 [-1.07, +1.23] |
| Faithfulness / H05 | +0.12 [-0.73, +1.00] | +0.11 [-0.66, +0.96] |
| Faithfulness / H06 | -0.52 [-1.15, +0.04] | -0.49 [-1.10, +0.05] |
| Faithfulness / H07 | -0.92 [-2.01, -0.02] | -0.86 [-1.95, -0.00] |
| Appropriateness proxy / H01 | +0.00 [-0.58, +0.59] | +0.01 [-0.55, +0.63] |
| Appropriateness proxy / H02 | +0.13 [-0.24, +0.72] | +0.12 [-0.26, +0.72] |
| Appropriateness proxy / H03 | +0.08 [-0.40, +0.96] | +0.08 [-0.40, +0.95] |
| Appropriateness proxy / H04 | +0.03 [-0.61, +0.99] | +0.03 [-0.62, +1.03] |
| Appropriateness proxy / H05 | -0.19 [-1.16, +0.21] | -0.19 [-1.13, +0.20] |
| Appropriateness proxy / H06 | -0.13 [-0.74, +0.22] | -0.16 [-0.78, +0.21] |
| Appropriateness proxy / H07 | +0.02 [-0.51, +0.75] | +0.02 [-0.51, +0.70] |
| Naturalness / H01 | -0.31 [-1.35, +0.14] | -0.35 [-1.53, +0.17] |
| Naturalness / H02 | -0.03 [-0.58, +0.41] | -0.08 [-0.69, +0.37] |
| Naturalness / H03 | +0.02 [-0.58, +0.81] | +0.05 [-0.65, +0.97] |
| Naturalness / H04 | -0.01 [-0.97, +0.82] | +0.00 [-1.01, +0.99] |
| Naturalness / H05 | +0.23 [-0.23, +1.40] | +0.29 [-0.24, +1.59] |
| Naturalness / H06 | -0.01 [-0.59, +0.55] | -0.03 [-0.69, +0.50] |
| Naturalness / H07 | +0.05 [-0.53, +0.93] | +0.04 [-0.59, +0.97] |
| Completeness / H01 | +0.30 [-0.49, +1.18] | +0.36 [-0.45, +1.24] |
| Completeness / H02 | +0.62 [+0.04, +1.22] | +0.61 [+0.02, +1.21] |
| Completeness / H03 | -0.55 [-1.31, +0.24] | -0.51 [-1.29, +0.28] |
| Completeness / H04 | -0.22 [-1.37, +1.07] | -0.25 [-1.41, +1.01] |
| Completeness / H05 | -0.31 [-1.06, +0.46] | -0.37 [-1.17, +0.43] |
| Completeness / H06 | +1.43 [+0.75, +2.23] | +1.47 [+0.76, +2.28] |
| Completeness / H07 | -1.32 [-2.16, -0.54] | -1.34 [-2.18, -0.53] |

::::::

### Interpretation

These are grading tendencies, not accuracy rankings. Intervals are pointwise, not simultaneous guarantees. One automatic judge cannot identify variation across automatic judges.

### Conclusion

- Human grading differences persist under both mappings for faithfulness, completeness.
- The table identifies which graders lie above or below the average human in both fits.

:::::::

:::::::{div}
:class: progress2-section progress2-q4

## Q4. Is disagreement concentrated in particular questions?

### Model

The paper adds item main effects and grader × item interactions. We estimate a shared question effect and an extra automatic effect for each question.


:::::{div}
:class: equation-sequence

**Paper specification** (notation simplified):

$$
\begin{aligned}
\phi_i={}&\beta_0+\beta_{1,g_i}+\beta_2x_i^{\mathrm{LLM}}\\
&+\beta_{3,q_i}+\beta_{4,g_i,q_i},\\
S_i&\sim\operatorname{OrderedLogistic}(\phi_i,\mathbf c).
\end{aligned}
$$
:::::

:::::{div}
:class: equation-sequence

**Separately fitted model for this question**, for each metric $m$:

$$
\begin{aligned}
\eta_{im}={}&B_{im}+\beta_{Am}A_i+\beta_{Lm}L_i+(1-A_i)h_{j(i)m}\\
&+u_{q(i)m}+A_iw_{q(i)m},\\
u_{qm}&=\sigma_{um}(\xi^u_{qm}-\bar\xi^u_m),\\
w_{qm}&=\sigma_{wm}(\xi^w_{qm}-\bar\xi^w_m),\\
\xi^u_{qm},\xi^w_{qm}&\overset{\mathrm{iid}}{\sim}\mathcal N(0,1),\quad
\sigma_{um},\sigma_{wm}\sim\operatorname{HalfNormal}(1).
\end{aligned}
$$

The rating likelihood is:

$$
Y_{im}\sim\operatorname{OrderedLogistic}(\eta_{im},\mathbf c_m).
$$
:::::

:::::{div}
:class: equation-sequence

**Quantity answering this question:**

$$
\begin{aligned}
\Delta_{AH,qm}&=\beta_{Am}+w_{qm},\\
\Delta_{AH,qm}-\Delta_{AH,m}&=w_{qm}.
\end{aligned}
$$
:::::

| Symbol | Meaning |
| --- | --- |
| $u_{qm}$ | Question effect shared by human and automatic ratings |
| $w_{qm}$ | Extra automatic shift for question $q$; plotted estimand |
| $\xi^u,\xi^w$ | Standardized latent effects, centered by their respective averages | 
| $i,m$ | Annotation and metric; each metric has its own coefficients and cutpoints |
| $Y_{im},\mathbf c_m$ | Category 1 to 4 and three ordered cutpoints |
| $A_i,L_i$ | Automatic = 1, human = 0; Llama = +0.5, GPT = −0.5 |
| $B_{im}$ | $\beta_{0m}+d_{\mathrm{domain}(i),m}+v_{\mathrm{conversation}(i),m}+r_{\mathrm{answer}(i),m}$ |

$B_{im}$ adjusts for domain and repeated annotations of answers within conversations. Human effects use $h_{jm}=\xi_{jm}-\bar\xi_m$, $\xi_{jm}\sim\mathcal N(0,1)$ with a fixed prior scale; Q3 instead estimates that scale.

### Results


:::::{div}
:class: research-plots

```{figure} figures/progress2/q4-items-faithfulness.svg
:alt: Faithfulness. Blue diamonds: the eight largest absolute posterior means per metric, with 95% credible intervals.
```
```{figure} figures/progress2/q4-items-appropriateness.svg
:alt: Appropriateness proxy. Blue diamonds: the eight largest absolute posterior means per metric, with 95% credible intervals.
```
```{figure} figures/progress2/q4-items-naturalness.svg
:alt: Naturalness. Blue diamonds: the eight largest absolute posterior means per metric, with 95% credible intervals.
```
```{figure} figures/progress2/q4-items-completeness.svg
:alt: Completeness. Blue diamonds: the eight largest absolute posterior means per metric, with 95% credible intervals.
```
:::::

Blue diamonds: the eight largest absolute posterior means per metric, with 95% credible intervals.

::::::{dropdown} Mapping counts


| Metric | Intervals excluding zero: primary | Alternative |
| --- | --- | --- |
| Faithfulness | 0 | 0 |
| Appropriateness proxy | 0 | 0 |
| Naturalness | 0 | 0 |
| Completeness | 0 | 0 |

::::::

::::::{dropdown} All 50 questions: descriptive gaps


```{figure} figures/progress2/all-questions.svg
:alt: Automatic score minus rescaled human mean, averaged over available answers. NA indicates no successful score. This is a descriptive plot, not a posterior estimate.

Automatic score minus rescaled human mean, averaged over available answers. NA indicates no successful score. This is a descriptive plot, not a posterior estimate.
```

::::::

### Interpretation

At most two candidate answers per question leave substantial uncertainty. Effects cannot independently establish intrinsic difficulty or generation reliability.

### Conclusion

- The current analysis does not clearly identify an additional automatic judge effect for any individual question: every 95% credible interval includes zero under both score mappings.
- This does not show that question effects are absent or that questions are equivalent.
- The available data and fitted model do not distinguish these effects clearly.

:::::::

:::::::{div}
:class: progress2-section progress2-q5

## Q5. Are longer responses scored differently?

### Model

The paper uses observed pairwise preferences. Our data contain absolute ratings, so we fit the stated ordinal extension with grader × length effects. The pairwise Binomial model is shown for reference and is not fitted.


:::::{div}
:class: equation-sequence

**Paper specification** (notation simplified):

$$
\begin{aligned}
y_i&\sim\operatorname{Binomial}(1,p_i),\\
\operatorname{logit}(p_i)&=\beta_0+\beta_{1,\pi_i}+\beta_{2,g_i}
+\beta_{3,g_i}x_i^{\mathrm{lengthDiff}},\\
\beta_{3,g}&\sim\mathcal N(\mu_{\mathrm{lengthDiff}},\sigma_{\mathrm{lengthDiff}}^2).
\end{aligned}
$$
:::::

:::::{div}
:class: equation-sequence

**Separately fitted model for this question**, for each metric $m$:

$$
\begin{aligned}
\eta_{im}={}&B_{im}+\beta_{Am}A_i+\beta_{Lm}L_i+(1-A_i)h_{j(i)m}\\
&+\beta_{zm}z_{im}+\beta_{Azm}A_i z_{im}.
\end{aligned}
$$

The rating likelihood is:

$$
Y_{im}\sim\operatorname{OrderedLogistic}(\eta_{im},\mathbf c_m).
$$
:::::

:::::{div}
:class: equation-sequence

**Quantity answering this question:**

$$
\begin{aligned}
\lambda_{H,m}&=\frac{\partial\eta_{im}}{\partial z_{im}}\bigg|_{A_i=0}=\beta_{zm},\\
\lambda_{A,m}&=\frac{\partial\eta_{im}}{\partial z_{im}}\bigg|_{A_i=1}=\beta_{zm}+\beta_{Azm},\\
\lambda_{A,m}-\lambda_{H,m}&=\beta_{Azm}.
\end{aligned}
$$
:::::

| Symbol | Meaning |
| --- | --- |
| $z_{im}$ | Standardized log(1 + word count), within each metric |
| $\lambda_{H,m},\lambda_{A,m}$ | Human and automatic length slopes |
| $\beta_{Azm}$ | Difference in slopes per one SD of log word count |
| $y_i,\pi_i,x_i^{\mathrm{lengthDiff}}$ in the paper | First answer wins, generator pair, token length difference | 
| $i,m$ | Annotation and metric; each metric has its own coefficients and cutpoints |
| $Y_{im},\mathbf c_m$ | Category 1 to 4 and three ordered cutpoints |
| $A_i,L_i$ | Automatic = 1, human = 0; Llama = +0.5, GPT = −0.5 |
| $B_{im}$ | $\beta_{0m}+d_{\mathrm{domain}(i),m}+v_{\mathrm{conversation}(i),m}+r_{\mathrm{answer}(i),m}$ |

$B_{im}$ adjusts for domain and repeated annotations of answers within conversations. Human effects use $h_{jm}=\xi_{jm}-\bar\xi_m$, $\xi_{jm}\sim\mathcal N(0,1)$ with a fixed prior scale; Q3 instead estimates that scale.

### Results


:::::{div}
:class: research-plots

```{figure} figures/progress2/q5-length-faithfulness.svg
:alt: Faithfulness. Green circles: human slope. Ochre squares: automatic slope. Blue diamonds: automatic minus human slope.
```
```{figure} figures/progress2/q5-length-appropriateness.svg
:alt: Appropriateness proxy. Green circles: human slope. Ochre squares: automatic slope. Blue diamonds: automatic minus human slope.
```
```{figure} figures/progress2/q5-length-naturalness.svg
:alt: Naturalness. Green circles: human slope. Ochre squares: automatic slope. Blue diamonds: automatic minus human slope.
```
```{figure} figures/progress2/q5-length-completeness.svg
:alt: Completeness. Green circles: human slope. Ochre squares: automatic slope. Blue diamonds: automatic minus human slope.
```
:::::

Green circles: human slope. Ochre squares: automatic slope. Blue diamonds: automatic minus human slope.

::::::{dropdown} Numerical slope differences


| Metric | Primary median [95% CrI] | Alternative median [95% CrI] |
| --- | --- | --- |
| Faithfulness | -0.99 [-1.67, -0.31] | -0.91 [-1.64, -0.25] |
| Appropriateness proxy | -0.05 [-0.65, +0.53] | +0.05 [-0.59, +0.66] |
| Naturalness | +0.48 [-0.05, +1.01] | +1.28 [+0.58, +2.03] |
| Completeness | -0.38 [-0.90, +0.14] | -0.11 [-0.68, +0.43] |

::::::

### Interpretation

A lower automatic slope does not mean its slope is negative. Length is an adjusted association, not causal verbosity bias. The data have no observed pairwise choices, so the paper’s Binomial model cannot be fitted directly.

### Conclusion

- **Faithfulness:** Lower automatic slope under both mappings.
- **Appropriateness proxy:** Difference unresolved under both mappings.
- **Naturalness:** Conclusion depends on mapping.
- **Completeness:** Difference unresolved under both mappings.

:::::::

::::::::
